In [1]:
class Student:
    def __init__(self, name, marks):
        self.name = name
        self.marks = marks

    def result(self):
        if self.marks >= 50:
            return "Pass"
        return "Fail"

student = Student("Ali", 72)
print(student.name)
print(student.result())

Ali
Pass


In [2]:
X = [
    ["Y", "N", "N", "N"],
    ["N", "Y", "N", "N"],
    ["Y", "Y", "N", "N"],
    ["Y", "N", "Y", "Y"],
    ["N", "Y", "Y", "N"]
]
y = ["-", "-", "+", "-", "+"]

print("X:", X)
print("y:", y)

X: [['Y', 'N', 'N', 'N'], ['N', 'Y', 'N', 'N'], ['Y', 'Y', 'N', 'N'], ['Y', 'N', 'Y', 'Y'], ['N', 'Y', 'Y', 'N']]
y: ['-', '-', '+', '-', '+']


In [3]:
def classification_error(y_true, y_pred):
    mistakes = 0
    for actual, predicted in zip(y_true, y_pred):
        if actual != predicted:
            mistakes += 1
    return mistakes / len(y_true)

y_true = ["+", "-", "+", "+"]
y_pred = ["+", "+", "+", "-"]
print(classification_error(y_true, y_pred)) 

0.5


In [4]:
from collections import Counter

class MajorityVoteClassifier:
    def __init__(self):
        self.majority_label = None

    def fit(self, X, y):
        counts = Counter(y)
        self.majority_label = counts.most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.majority_label for _ in X]

In [5]:
X_train = [
    ["Y", "N"],
    ["N", "Y"],
    ["Y", "Y"],
    ["Y", "N"],
    ["N", "Y"]
]
y_train = ["-", "-", "+", "-", "+"]

model = MajorityVoteClassifier()
model.fit(X_train, y_train)

X_test = [["Y", "Y"], ["N", "N"]]
print(model.predict(X_test))  

['-', '-']


In [6]:
import random

class MemorizerClassifier:
    def __init__(self):
        self.memory = {}
        self.labels = []

    def fit(self, X, y):
        self.memory = {}
        self.labels = list(set(y))
        for features, label in zip(X, y):
            self.memory[tuple(features)] = label
        return self

    def predict(self, X):
        predictions = []
        for features in X:
            key = tuple(features)
            if key in self.memory:
                predictions.append(self.memory[key])
            else:
                predictions.append(random.choice(self.labels))
        return predictions

In [7]:
model = MemorizerClassifier()
model.fit(X_train, y_train)
print(model.predict(X_train))  

['-', '+', '+', '-', '+']


In [8]:
from collections import defaultdict, Counter

class DecisionStumpClassifier:
    def __init__(self, feature_index=0):
        self.feature_index = feature_index
        self.rules = {}
        self.default_label = None

    def fit(self, X, y):
        groups = defaultdict(list)
        for features, label in zip(X, y):
            value = features[self.feature_index]
            groups[value].append(label)

        self.rules = {}
        for value, labels in groups.items():
            self.rules[value] = Counter(labels).most_common(1)[0][0]

        self.default_label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        predictions = []
        for features in X:
            value = features[self.feature_index]
            predictions.append(self.rules.get(value, self.default_label))
        return predictions

In [9]:
stump = DecisionStumpClassifier(feature_index=0)
stump.fit(X, y)
print(stump.rules)
print(stump.predict(X))

{'Y': '-', 'N': '-'}
['-', '-', '-', '-', '-']


In [10]:
X_train = [
    ["Y", "N", "N", "N"],
    ["N", "Y", "N", "N"],
    ["Y", "Y", "N", "N"],
    ["Y", "N", "Y", "Y"],
    ["N", "Y", "Y", "N"]
]
y_train = ["-", "-", "+", "-", "+"]

X_test = [
    ["Y", "Y", "Y", "N"],
    ["N", "N", "N", "N"],
    ["Y", "N", "N", "Y"]
]
y_test = ["+", "-", "-"]

In [11]:
majority = MajorityVoteClassifier()
majority.fit(X_train, y_train)

train_pred = majority.predict(X_train)
test_pred = majority.predict(X_test)

train_error = classification_error(y_train, train_pred)
test_error = classification_error(y_test, test_pred)

print("Majority Vote")
print("Training error:", train_error)
print("Test error:", test_error)

Majority Vote
Training error: 0.4
Test error: 0.3333333333333333


In [12]:
memorizer = MemorizerClassifier()
memorizer.fit(X_train, y_train)

train_pred = memorizer.predict(X_train)
print("Memorizer training error:", classification_error(y_train, train_pred))
assert classification_error(y_train, train_pred) == 0.0, "Memorizer should have zero training error"

test_pred = memorizer.predict(X_test)
print("Memorizer predictions on unseen X_test:", test_pred)
print("Memorizer test error:", classification_error(y_test, test_pred))

Memorizer training error: 0.0
Memorizer predictions on unseen X_test: ['+', '-', '+']
Memorizer test error: 0.3333333333333333


In [13]:
for idx in range(len(X_train[0])):
    stump = DecisionStumpClassifier(feature_index=idx)
    stump.fit(X_train, y_train)
    train_pred = stump.predict(X_train)
    err = classification_error(y_train, train_pred)
    print(f"feature_index={idx} -> rules={stump.rules}, training error={err}")

feature_index=0 -> rules={'Y': '-', 'N': '-'}, training error=0.4
feature_index=1 -> rules={'N': '-', 'Y': '+'}, training error=0.2
feature_index=2 -> rules={'N': '-', 'Y': '-'}, training error=0.4
feature_index=3 -> rules={'N': '-', 'Y': '-'}, training error=0.4


In [14]:
stump = DecisionStumpClassifier(feature_index=2)
stump.fit(X_train, y_train)

models = {
    "Majority Vote": majority,
    "Memorizer": memorizer,
    "Decision Stump": stump
}

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_error = classification_error(y_train, train_pred)
    test_error = classification_error(y_test, test_pred)
    print(name)
    print("Training error:", train_error)
    print("Test error:", test_error)
    print()

Majority Vote
Training error: 0.4
Test error: 0.3333333333333333

Memorizer
Training error: 0.0
Test error: 0.3333333333333333

Decision Stump
Training error: 0.4
Test error: 0.3333333333333333



In [15]:
class BaseClassifier:
    def fit(self, X, y):
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

    def score(self, X, y):
        predictions = self.predict(X)
        correct = sum(actual == predicted for actual, predicted in zip(y, predictions))
        return correct / len(y)


class MajorityVoteClassifierV2(BaseClassifier):
    def __init__(self):
        self.majority_label = None

    def fit(self, X, y):
        self.majority_label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.majority_label for _ in X]


class MemorizerClassifierV2(BaseClassifier):
    def __init__(self):
        self.memory = {}
        self.labels = []

    def fit(self, X, y):
        self.memory = {}
        self.labels = list(set(y))
        for features, label in zip(X, y):
            self.memory[tuple(features)] = label
        return self

    def predict(self, X):
        predictions = []
        for features in X:
            key = tuple(features)
            predictions.append(self.memory.get(key, random.choice(self.labels)))
        return predictions


class DecisionStumpClassifierV2(BaseClassifier):
    def __init__(self, feature_index=0):
        self.feature_index = feature_index
        self.rules = {}
        self.default_label = None

    def fit(self, X, y):
        groups = defaultdict(list)
        for features, label in zip(X, y):
            groups[features[self.feature_index]].append(label)
        self.rules = {v: Counter(labels).most_common(1)[0][0] for v, labels in groups.items()}
        self.default_label = Counter(y).most_common(1)[0][0]
        return self

    def predict(self, X):
        return [self.rules.get(f[self.feature_index], self.default_label) for f in X]

In [16]:
models_v2 = {
    "Majority Vote": MajorityVoteClassifierV2(),
    "Memorizer": MemorizerClassifierV2(),
    "Decision Stump (feat=2)": DecisionStumpClassifierV2(feature_index=2)
}

for name, m in models_v2.items():
    m.fit(X_train, y_train)
    print(f"{name}: train accuracy = {m.score(X_train, y_train):.3f}, test accuracy = {m.score(X_test, y_test):.3f}")

Majority Vote: train accuracy = 0.600, test accuracy = 0.667
Memorizer: train accuracy = 1.000, test accuracy = 0.333
Decision Stump (feat=2): train accuracy = 0.600, test accuracy = 0.667


In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.head())
print(train.shape)
print(test.shape)
print(train.info())
print(train["Survived"].value_counts())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

In [ ]:
features = ["Pclass", "Sex", "Age", "Fare"]

X_full = train[features].copy()
y_full = train["Survived"].copy()
X_test_full = test[features].copy()

for df in (X_full, X_test_full):
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
    df["Age"] = df["Age"].fillna(train["Age"].median())
    df["Fare"] = df["Fare"].fillna(train["Fare"].median())

print(X_full.head())

   Pclass  Sex   Age     Fare
0       3    0  22.0   7.2500
1       1    1  38.0  71.2833
2       3    1  26.0   7.9250
3       1    1  35.0  53.1000
4       3    0  35.0   8.0500


In [ ]:
majority = MajorityVoteClassifier()
majority.fit(X_train_split, y_train_split)

train_predictions = majority.predict(X_train_split)
valid_predictions = majority.predict(X_valid_split)

print("Majority Vote training accuracy:", 1 - classification_error(y_train_split, train_predictions))
print("Majority Vote validation accuracy:", 1 - classification_error(y_valid_split, valid_predictions))

Majority Vote training accuracy: 0.6165730337078652
Majority Vote validation accuracy: 0.6145251396648045


In [ ]:
from sklearn.model_selection import train_test_split as _tts

X_train_num, X_valid_num, y_train_num, y_valid_num = _tts(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

X_train_exact = X_train_num.astype(str).values.tolist()
X_valid_exact = X_valid_num.astype(str).values.tolist()
y_train_exact = y_train_num.astype(str).tolist()
y_valid_exact = y_valid_num.astype(str).tolist()

memorizer = MemorizerClassifier()
memorizer.fit(X_train_exact, y_train_exact)

train_pred = memorizer.predict(X_train_exact)
valid_pred = memorizer.predict(X_valid_exact)

print("Memorizer training accuracy:", 1 - classification_error(y_train_exact, train_pred))
print("Memorizer validation accuracy:", 1 - classification_error(y_valid_exact, valid_pred))

Memorizer training accuracy: 0.976123595505618
Memorizer validation accuracy: 0.6089385474860336


In [ ]:
feature_names = ["Pclass", "Sex", "Age", "Fare"]

for feature_index, feature_name in enumerate(feature_names):
    stump = DecisionStumpClassifier(feature_index=feature_index)
    stump.fit(X_train_split, y_train_split)
    predictions = stump.predict(X_train_split)
    accuracy = 1 - classification_error(y_train_split, predictions)
    print(feature_name, "training accuracy:", accuracy)

Pclass training accuracy: 0.6882022471910112
Sex training accuracy: 0.7893258426966292
Age training accuracy: 0.6306179775280899
Fare training accuracy: 0.648876404494382


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train_num, y_train_num)
predictions = tree.predict(X_valid_num)
print("Decision Tree validation accuracy:", (predictions == y_valid_num).mean())


Decision Tree validation accuracy: 0.7932960893854749


In [ ]:
final_model = DecisionTreeClassifier(max_depth=3, random_state=42)
final_model.fit(X_full, y_full)
test_predictions = final_model.predict(X_test_full)

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(submission.shape)  

   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
(418, 2)
